In [2]:
# pip install torchvision

In [3]:
import torch
import os
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image

In [ ]:
# image load => transform -> dataset
class imageProcessor:
    def __init__(self, root_dir_path, transformation=None):
        self.root_dir_path = root_dir_path
        self.transformations = transformations

        # list of path
        self.all_img_paths = os.path.join(root_dir_path, img) in for img in os.listdir(root_dir_path)

    def __len__(self):
        return len(self.all_img_paths)

    def __getitem__(self, idx):
        img_path = self.all_img_paths[idx]
        img = Image.open(img_path).convert("RGB")

        if self.transformations:
            img = self.transformations(img)

        return img

In [ ]:
root_dir_path = "./imag_align_celeba"

transformations = transforms.Compose([
    transforms.CenterCrop(178),
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),(0.5, 0.5, 0.5)) #[-1.1]
])

In [ ]:
dataset = ImagePRocessor(root_dir_path, transformations)
print(f"loaded {len(datasets)}, images")

In [ ]:
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

# Generator Network

In [ ]:
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim = 100, img_channels=3):  #img_channels=3 for rgb
        super(Generator, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.ReLU(),
            
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, 64*64*img_channels),
            nn.Tanh(),
        )


    def forward(self, z):
        img = self.model(z)
        img = img.view(img.zise(0), 3, 64, 64)
        return img
        

# Discriminator Network

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, img_channels=3):  #img_channels=3 for rgb
        super(Discriminator, self).__init__()

        self.model = nn.Sequential(
            nn.Flatten(),   #4D to 1D
            
            nn.Linear(264*64*img_channels, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Linear(256, 1),
            nn.sigmoid()    #probability
        )

    def forward(self, img):
        return self.model(img
        

In [ ]:
GAN_loss = nn.BCELoss()

generator = Generator()
g_optimizer = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5,0.999)) 

discriminator = Discriminator()
g_optimizer = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5,0.999)) 


In [ ]:
# device
if torch.backeneds.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else :
    device = torch.device("cpu")

print(f"device is {device})

In [ ]:
generator = generator.to(device)
discriminator = discriminator.to(device)

# GAN Training

In [ ]:
def train(generator, discriminator, dataloader, epochs=10):
    for epoch in range(epochs):
        for i, imgs in enumerate(dataloader):
            real_img = imgs.to(device)

            #Create labels
            real_labels = torch.ones(batch_size, 1).to(device)
            fake_labels = torch.zeros(batch_size, 1).to(device)

            # Train D
            d_optimizer.zero_grad()
            fake_imgs = generator(torch.randn(batch_size, 100)).to(device)
            
            real_loss = GAN_loss(discriminator(real_imgs),real_labels) 
            fale_loss = GAN_loss(discriminator(fake_imgs.detach()), fake_labels)

            d_loss = (real_loss + fake_loss)/2

            d_loss.backward()
            d_optimizer.step()

            # Train G
            g_optimizer.zero_grad()
            g_loss = GAN_loss(discriminator(fake_imgs), real_labels)

            g_loss.backward()
            g_optimizer.step()

            if i % 50 == 0:
                print(f"for epoch : {epoch+1}/{epochs} batch : {i+1} G-loss: {g_loss} D-loss: {d_loss}")

            save_generated_images(generator, epoch, device)
            

In [ ]:
import matplotlib.pyplot as plt
import torchvision

def save_generated_images(generator, epoch, device, num_image=8):
    z = torch.randn(num_imgs, 100).to(device)
    generated_imgs = generator(z).detach().cpu()

    grid = torchvision.utils.make_grid(generated_imgs, nrows=4, normalize=True)

    plt.imshow(np.transpose(grid, (1,2,0))
    plt.title(f"epoch {epoch+1}")
    plt.axis("off")
    plt.show()
    

In [ ]:
train(generator, discriminaator, dataloader, epochs=5)